## ⚠ Save a copy to your Drive first

**This notebook is fetched fresh from GitHub every time you open the link.** Any edits you make here — settings, code, hyperparameters — **will be LOST when you close the tab** unless you save a copy.

**To keep your edits:**

1. **File → Save a copy in Drive** (top menu)
2. Re-open the saved copy via **File → Open notebook → Recent** or your Google Drive next time

The saved copy is yours to edit; the GitHub link always opens fresh.

# Train an Image Detector for IGNODE

**Maintained by:** IGNODE  
**Last verified:** June 2026 against YOLOX 0.3.0, PyTorch 2.4, ONNX 1.16  
**Runtime:** ~15-45 minutes on Colab's free T4 GPU; impractical on CPU

Train an image-detection model — predict bounding boxes + class labels for objects in an image (e.g. fruit on a conveyor, defects on a part, vehicles in a parking lot). Uses Megvii YOLOX with COCO-pretrained backbones (nano / tiny / s / m / l / x). Output is byte-compatible with IGNODE's in-platform detection trainer — same ONNX shape, same sidecar contract.

## ⚙ Settings — set these first, then Runtime → Run all

**You only need to edit one thing**: how to give the notebook a dataset. Everything else has sensible defaults you can tweak later.

### Two ways to give the notebook a dataset

**Way 1 — Drag-drop a file (leave `DATASET_URL = ''`):**

1. Click the folder icon (📁) on the far-left vertical toolbar to open Colab's **Files panel**.
2. Drag your `.zip`, `.tar`, or `.tar.gz` from your laptop onto the Files panel — it appears under `/content/`.
3. Run all cells from the top (**Runtime → Run all**). The Load Dataset step (§3) automatically picks up whatever you dropped.

**Way 2 — Paste a download URL (set `DATASET_URL`):**

Any HTTPS link that returns the archive on GET. Common sources:
- **IGNODE Uploads page → Download (↓ icon) → set TTL=60 min → Generate URL → Copy.** Paste between the quotes in the cell below.
- **Azure Blob SAS URL** (signed Azure Storage link).
- **Google Drive direct-download link** (must be the `?export=download` form, NOT the share-preview URL).
- **Ultralytics test datasets** (handy to try the notebook without your own data):
  - `https://github.com/ultralytics/yolov5/releases/download/v1.0/coco8.zip` (8 images, ~1 MB — plumbing test only; mAP will be near 0)
  - `https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip` (128 images, ~7 MB)

### Accepted archive formats

| Container | Inside the archive |
|---|---|
| `.zip` / `.tar` / `.tar.gz` | One of the three layouts below |

The notebook auto-detects which layout is inside, then converts to YOLOX's expected COCO directory shape (Steps 4 + 5).

**Layout A — Roboflow COCO** (recommended; what Roboflow Universe exports by default when you pick "COCO"):

```
your-dataset.zip
├── train/
│   ├── _annotations.coco.json     ← required; supervision reads this
│   ├── img_001.jpg
│   ├── img_002.jpg
│   └── ...
├── valid/
│   ├── _annotations.coco.json
│   └── img_VVV.jpg ...
└── test/                          ← optional; merged into val for stronger mAP
    ├── _annotations.coco.json
    └── img_TTT.jpg ...
```

**Layout B — Roboflow YOLO / Ultralytics YOLOv8** (what Roboflow exports when you pick "YOLOv8"):

```
your-dataset.zip
├── data.yaml                      ← required; lists class names
├── train/
│   ├── images/
│   │   └── img_001.jpg ...
│   └── labels/
│       └── img_001.txt            ← YOLO normalized boxes
├── valid/
│   ├── images/
│   └── labels/
└── test/                          ← optional
    ├── images/
    └── labels/
```

**Layout C — MS-COCO 2017** (custom pipelines, CVAT export, LabelStudio export):

```
your-dataset.tar
├── annotations/
│   ├── instances_train2017.json   ← required
│   └── instances_val2017.json
├── train2017/                     ← OR images/train/ — sniffer accepts both
│   └── img_001.jpg ...
└── val2017/
    └── img_VVV.jpg ...
```

If your data is in another format (Pascal VOC, raw CSV, etc.), convert to one of the three above first.

### Training knobs (sensible defaults; tune later if you want)

- `BACKBONE` — `yolox_nano` / `yolox_tiny` / `yolox_s` / `yolox_m` / `yolox_l` / `yolox_x`. Default `yolox_s` is the sweet spot on a free T4. Smaller = less likely to overfit on tiny datasets. Larger = better accuracy if you have more data + VRAM.
- `EPOCHS` — passes through the training set. Detection needs 50-100 for real mAP; below 30 it usually stays at 0.
- `NO_AUG_EPOCHS` — disable mosaic augmentation in the last N epochs so the model sees real (non-composited) images before training ends. Critical for small data.
- `BATCH_SIZE` — drop to 8 if you hit CUDA OOM on a larger backbone.

In [ ]:
# ════════════════════ EDIT THIS ════════════════════
DATASET_URL = ''   # ← paste IGNODE presigned / SAS / GDrive / HTTPS URL, OR leave empty for drag-drop
# ═══════════════════════════════════════════════════

# ─── Training knobs (sensible defaults) ───
BACKBONE       = 'yolox_s'   # nano / tiny / s / m / l / x
EPOCHS         = 60
NO_AUG_EPOCHS  = 20          # last N epochs without mosaic
BATCH_SIZE     = 16
IMG_SIZE       = 640         # YOLOX standard
MIXUP_PROB     = 1.0
MOSAIC_PROB    = 1.0
EXPERIMENT_NAME = 'ignode_detector'
BASIC_LR       = 0.01 / 64   # YOLOX scales by batch size internally

_BACKBONE_DIMS = {
    'yolox_nano': (0.33, 0.25),
    'yolox_tiny': (0.33, 0.375),
    'yolox_s':    (0.33, 0.50),
    'yolox_m':    (0.67, 0.75),
    'yolox_l':    (1.00, 1.00),
    'yolox_x':    (1.33, 1.25),
}
if BACKBONE not in _BACKBONE_DIMS:
    raise ValueError(f'BACKBONE must be one of {list(_BACKBONE_DIMS)}, got {BACKBONE!r}')
DEPTH, WIDTH = _BACKBONE_DIMS[BACKBONE]

if DATASET_URL:
    print(f'Dataset:  will download from URL ({len(DATASET_URL)} chars)')
else:
    print('Dataset:  drag-drop mode (upload widget appears in §3)')
print(f'Backbone: {BACKBONE}  (depth={DEPTH}, width={WIDTH})')
print(f'Schedule: {EPOCHS} epochs  (last {NO_AUG_EPOCHS} without mosaic)')
print(f'Batch:    {BATCH_SIZE}  Image: {IMG_SIZE}x{IMG_SIZE}')

## Quick start

1. **Runtime → Change runtime type → GPU** (T4 is fine; free tier)
2. Set `DATASET_URL` in the Settings cell above, OR leave it empty to drag-drop your archive into Colab's Files panel
3. **Runtime → Run all**
4. Wait for training (~15-45 min)
5. The last cell automatically downloads `model.onnx` + sidecar JSONs to your laptop

### What you get at the end

- `model.onnx` — your trained YOLOX detector
- `class_labels.json` — class names (in the order the model emits them)
- `preprocess_config.json` — input size, letterbox config, channel order. IGNODE's inference runtime reads this to preprocess images at predict time the way we did at training time.

Drop the artifacts into **IGNODE → ML Factory → Custom Models → + Upload ML Model**.

## 1. Install pinned dependencies

Packages:
- `yolox` — Megvii YOLOX detection trainer (Apache 2.0); same engine IGNODE's in-platform detection trainer uses
- `supervision` — dataset format normalization (Roboflow COCO / YOLO → YOLOX-expected COCO layout)
- `pycocotools` — YOLOX's annotation reader
- `onnx` + `onnxruntime` — export + sanity check

PyTorch is pre-installed in Colab. The `--no-deps` on yolox is intentional — it ships an over-pinned `onnx==1.13` that fights with the newer `onnx` we want for the export.

In [ ]:
!pip install -q supervision==0.23.0 pycocotools onnx onnxruntime onnxscript loguru tabulate thop
!pip install -q --no-deps yolox==0.3.0

import sys, torch, supervision as sv, onnx, onnxruntime, yolox
print(f'Python:       {sys.version.split()[0]}')
print(f'PyTorch:      {torch.__version__}')
print(f'YOLOX:        {yolox.__version__}')
print(f'supervision:  {sv.__version__}')
print(f'onnx:         {onnx.__version__}')
print(f'onnxruntime:  {onnxruntime.__version__}')

## 2. Check the runtime

Detection training is impractical without a GPU. The cell below confirms one is wired. If it reports CPU, go to **Runtime → Change runtime type → GPU** (T4 is fine; available for free) and **Restart runtime**, then re-run from the top.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    name = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU available: {name}  ({mem:.1f} GB VRAM)')
    if mem < 14:
        print('⚠  Less than 14 GB VRAM — stick with yolox_nano or yolox_tiny backbones (Step 5).')
else:
    DEVICE = torch.device('cpu')
    print('❌ CPU only — YOLOX training is impractical on CPU (>10× slower than T4).')
    print('   Runtime → Change runtime type → GPU → Save → Runtime → Restart runtime, then re-run from the top.')

## 3. Load dataset

Uses `DATASET_URL` from the Settings cell at the top. If empty, an interactive upload widget appears below — drag your archive into Colab's Files panel BEFORE running this cell, or use the upload widget that pops up.

In [ ]:
# Single dataset-load cell — branches on DATASET_URL from the Settings cell above
import os, shutil, glob
RAW_DIR = '/content/dataset_raw'
shutil.rmtree(RAW_DIR, ignore_errors=True)
os.makedirs(RAW_DIR, exist_ok=True)

if DATASET_URL:
    lower = DATASET_URL.lower()
    if '.zip' in lower:
        out_name = 'dataset.zip'
    elif '.tar.gz' in lower or '.tgz' in lower:
        out_name = 'dataset.tar.gz'
    else:
        out_name = 'dataset.tar'
    ARCHIVE_PATH = os.path.join(RAW_DIR, out_name)
    get_ipython().system('wget -q --show-progress -O "$ARCHIVE_PATH" "$DATASET_URL"')
    size_mb = os.path.getsize(ARCHIVE_PATH) / (1024 * 1024)
    print(f'Downloaded {ARCHIVE_PATH}  ({size_mb:.1f} MB)')
else:
    # Drag-drop fallback. Look first for an archive the customer dropped
    # under /content/ BEFORE running (the most natural Files-panel flow);
    # only pop the upload widget if nothing's already there.
    from google.colab import files
    archives_root = (glob.glob('/content/*.zip') + glob.glob('/content/*.tar')
                     + glob.glob('/content/*.tar.gz') + glob.glob('/content/*.tgz'))
    if archives_root:
        ARCHIVE_PATH = archives_root[0]
        print(f'Found pre-dropped {ARCHIVE_PATH}')
    else:
        uploaded = files.upload()
        for name, data in uploaded.items():
            with open(os.path.join(RAW_DIR, name), 'wb') as f: f.write(data)
        archives = (glob.glob(f'{RAW_DIR}/*.zip') + glob.glob(f'{RAW_DIR}/*.tar')
                    + glob.glob(f'{RAW_DIR}/*.tar.gz') + glob.glob(f'{RAW_DIR}/*.tgz'))
        if not archives:
            raise FileNotFoundError('No .zip / .tar archive uploaded.')
        ARCHIVE_PATH = archives[0]
        print(f'Uploaded {ARCHIVE_PATH}')

print(f'\nARCHIVE_PATH = {ARCHIVE_PATH}')

## 4. Extract + auto-detect format

Reads whatever Roboflow exported (COCO / YOLOv8) or your own MS-COCO bundle, drops macOS metadata noise, and normalizes the on-disk layout into what YOLOX expects. Uses `supervision` for the heavy lifting — same library IGNODE's format-converter uses internally.

In [ ]:
import os, shutil, zipfile, tarfile, glob
EXTRACT_DIR = '/content/dataset_extracted'
shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

if zipfile.is_zipfile(ARCHIVE_PATH):
    print('Archive: ZIP')
    with zipfile.ZipFile(ARCHIVE_PATH) as z:
        z.extractall(EXTRACT_DIR)
elif tarfile.is_tarfile(ARCHIVE_PATH):
    print('Archive: TAR')
    with tarfile.open(ARCHIVE_PATH) as t:
        t.extractall(EXTRACT_DIR)
else:
    raise ValueError(f'Not a ZIP or TAR: {ARCHIVE_PATH}')

# Sweep macOS metadata + .DS_Store — they crash supervision's COCO loader
for junk in glob.glob(f'{EXTRACT_DIR}/**/__MACOSX', recursive=True):
    shutil.rmtree(junk, ignore_errors=True)
for junk in glob.glob(f'{EXTRACT_DIR}/**/.DS_Store', recursive=True):
    os.remove(junk)

print('\nTop-level extracted contents:')
for entry in sorted(os.listdir(EXTRACT_DIR))[:20]:
    p = os.path.join(EXTRACT_DIR, entry)
    label = 'DIR ' if os.path.isdir(p) else 'FILE'
    print(f'  {label}  {entry}')

In [ ]:
# IR-3.L (cleanup) — let supervision decide what it can parse.
#
# Discovery walks the extracted tree for the things supervision needs:
#   - COCO path: any *.coco.json or instances_*.json file, plus the
#     images directory siblling/nested next to it.
#   - YOLO path: any directory named 'labels' or 'labels/<split>' with
#     .txt files, plus a sibling 'images' (or 'images/<split>') dir.
# Whichever is found per-split is what we hand to supervision. No
# typed SOURCE_FORMAT enum, no 4-way branching — there are really only
# two supervision call shapes.
#
# Per-split search uses split keywords (['train'] for train,
# ['valid', 'val', 'test'] for val). If a YOLO archive ships without
# a val split (coco128's case), val falls back to train so the
# pipeline runs end-to-end — useful for sanity-check 'does the
# pipeline work at all?' runs.

import os, re, glob, yaml
import supervision as sv

def discover_split(extract_dir, keywords):
    """Return a dict describing how to load one split, or None if not found.

    Returned dict shape:
      {'mode': 'coco', 'json': <path>, 'images': <dir>}
      {'mode': 'yolo', 'labels': <dir>, 'images': <dir>}
    """
    # COCO patterns — try first since they're unambiguous
    for kw in keywords:
        # Roboflow COCO: <kw>/_annotations.coco.json
        for j in glob.glob(f'{extract_dir}/**/{kw}/_annotations.coco.json', recursive=True):
            return {'mode': 'coco', 'json': j, 'images': os.path.dirname(j)}
        # MS-COCO: annotations/instances_<kw>*.json
        for j in glob.glob(f'{extract_dir}/**/instances_{kw}*.json', recursive=True):
            ann_dir = os.path.dirname(j)
            root    = os.path.dirname(ann_dir)
            m = re.match(r'instances_(\w+)\.json', os.path.basename(j))
            split_name = m.group(1) if m else kw
            img_dir = os.path.join(root, split_name)
            if os.path.isdir(img_dir):
                return {'mode': 'coco', 'json': j, 'images': img_dir}

    # YOLO patterns
    for kw in keywords:
        # Pattern A — Roboflow YOLO: <kw>/labels + <kw>/images
        for labels_dir in glob.glob(f'{extract_dir}/**/{kw}/labels', recursive=True):
            if not glob.glob(f'{labels_dir}/*.txt'): continue
            img_dir = os.path.join(os.path.dirname(labels_dir), 'images')
            if os.path.isdir(img_dir):
                return {'mode': 'yolo', 'labels': labels_dir, 'images': img_dir}
        # Pattern B — Ultralytics YOLO: labels/<kw>*/ + images/<kw>*/
        for labels_dir in glob.glob(f'{extract_dir}/**/labels/{kw}*', recursive=True):
            if not os.path.isdir(labels_dir): continue
            if not glob.glob(f'{labels_dir}/*.txt'): continue
            split_name = os.path.basename(labels_dir)
            parent     = os.path.dirname(os.path.dirname(labels_dir))
            img_dir    = os.path.join(parent, 'images', split_name)
            if os.path.isdir(img_dir):
                return {'mode': 'yolo', 'labels': labels_dir, 'images': img_dir}
    return None

# Discover both splits.
train_inputs = discover_split(EXTRACT_DIR, ['train'])
val_inputs   = discover_split(EXTRACT_DIR, ['valid', 'val', 'test'])

if train_inputs is None:
    raise ValueError(
        'Could not find any training split. Expected one of:\n'
        '  - <split>/_annotations.coco.json   (Roboflow COCO)\n'
        '  - annotations/instances_train*.json (MS-COCO)\n'
        '  - train/labels/*.txt + train/images/   (Roboflow YOLO)\n'
        '  - labels/train*/*.txt + images/train*/  (Ultralytics YOLO)'
    )

if val_inputs is None:
    print('No val split found — using train as val (mAP on training data; sanity-check only)')
    val_inputs = train_inputs

print(f'Train: mode={train_inputs["mode"]:5}  images={train_inputs["images"]}')
print(f'Val:   mode={val_inputs["mode"]:5}  images={val_inputs["images"]}')

# Stash for the converter cell below.
DATASET_INPUTS = {'train': train_inputs, 'val': val_inputs}


## 5. Convert dataset to YOLOX layout

YOLOX expects MS-COCO directory layout:

```
/content/coco/
  annotations/
    instances_train2017.json
    instances_val2017.json
  train2017/<image>.jpg
  val2017/<image>.jpg
```

Roboflow COCO and Roboflow YOLO both get normalized into this shape. If your `valid/` and `test/` splits are small, the test set is merged into validation to give mAP more signal (typical for under-500-image datasets).

In [ ]:
# Convert via supervision — two call shapes total, no typed format enum.
import os, shutil, glob, json, yaml
from collections import Counter
import supervision as sv

COCO_DIR = '/content/coco'
shutil.rmtree(COCO_DIR, ignore_errors=True)
os.makedirs(f'{COCO_DIR}/annotations', exist_ok=True)
os.makedirs(f'{COCO_DIR}/train2017', exist_ok=True)
os.makedirs(f'{COCO_DIR}/val2017',   exist_ok=True)

# For YOLO mode, supervision needs a data.yaml to bind class names.
# Try first to find one already in the extracted tree; otherwise
# synthesize from the max class index across all label files.
def find_or_synth_yaml(extract_dir, label_dirs):
    yaml_candidates = glob.glob(f'{extract_dir}/**/data.yaml', recursive=True)
    if yaml_candidates:
        return yaml_candidates[0]
    # Synthesize
    max_class = -1
    for d in label_dirs:
        for lbl in glob.glob(f'{d}/*.txt'):
            with open(lbl) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        try: max_class = max(max_class, int(parts[0]))
                        except ValueError: pass
    n = max_class + 1 if max_class >= 0 else 1
    # COCO-80 names if exactly 80 classes (covers coco128 + similar);
    # generic class_N otherwise. Class names only affect display, not
    # learning quality.
    coco80 = ['person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball','kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair','couch','potted plant','bed','dining table','toilet','tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven','toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear','hair drier','toothbrush']
    names = coco80 if n == 80 else [f'class_{i}' for i in range(n)]
    out = '/tmp/_synth_data.yaml'
    with open(out, 'w') as f:
        yaml.safe_dump({'path': extract_dir, 'train': '', 'val': '',
                        'names': {i: n for i, n in enumerate(names)},
                        'nc': len(names)}, f)
    return out

# Build the synth/found data.yaml once if any YOLO splits are present.
yolo_label_dirs = [inputs['labels'] for inputs in DATASET_INPUTS.values()
                    if inputs['mode'] == 'yolo']
data_yaml = find_or_synth_yaml(EXTRACT_DIR, yolo_label_dirs) if yolo_label_dirs else None

# Load + write each split. Two supervision call shapes — picked
# directly by inputs['mode']. No format type enum, no 4-way branching.
for split_name, inputs, dst_dir, dst_json in [
    ('train', DATASET_INPUTS['train'], 'train2017', 'instances_train2017.json'),
    ('val',   DATASET_INPUTS['val'],   'val2017',   'instances_val2017.json'),
]:
    if inputs['mode'] == 'coco':
        ds = sv.DetectionDataset.from_coco(
            images_directory_path=inputs['images'],
            annotations_path=inputs['json'],
        )
    else:  # yolo
        ds = sv.DetectionDataset.from_yolo(
            images_directory_path=inputs['images'],
            annotations_directory_path=inputs['labels'],
            data_yaml_path=data_yaml,
        )
    ds.as_coco(
        images_directory_path=f'{COCO_DIR}/{dst_dir}',
        annotations_path=f'{COCO_DIR}/annotations/{dst_json}',
    )

# Drop phantom classes (declared but never annotated).
with open(f'{COCO_DIR}/annotations/instances_train2017.json') as f: t = json.load(f)
with open(f'{COCO_DIR}/annotations/instances_val2017.json')   as f: v = json.load(f)
active = {a['category_id'] for a in t['annotations']} | {a['category_id'] for a in v['annotations']}
if any(c['id'] not in active for c in t['categories']):
    for d in (t, v):
        d['categories'] = [c for c in d['categories'] if c['id'] in active]
    remap = {c['id']: i + 1 for i, c in enumerate(sorted(t['categories'], key=lambda x: x['id']))}
    for d in (t, v):
        for c in d['categories']: c['id'] = remap[c['id']]
        for a in d['annotations']: a['category_id'] = remap[a['category_id']]
    with open(f'{COCO_DIR}/annotations/instances_train2017.json', 'w') as f: json.dump(t, f)
    with open(f'{COCO_DIR}/annotations/instances_val2017.json',   'w') as f: json.dump(v, f)

CLASSES = [c['name'] for c in sorted(t['categories'], key=lambda x: x['id'])]
NUM_CLASSES = len(CLASSES)
print(f'\nFinal layout at {COCO_DIR}')
print(f'  train2017/: {len(t["images"])} images, {len(t["annotations"])} boxes')
print(f'  val2017/:   {len(v["images"])} images, {len(v["annotations"])} boxes')
print(f'  classes ({NUM_CLASSES}): {CLASSES[:10]}{"…" if NUM_CLASSES > 10 else ""}')


## 6. Inspect the dataset

Per-class box counts in train + val. If a class has < 10 val boxes its mAP is going to be noise-dominated regardless of training quality — not a notebook bug, that's just the metric.

In [ ]:
import json
from collections import Counter
from tabulate import tabulate

with open(f'{COCO_DIR}/annotations/instances_train2017.json') as f: t = json.load(f)
with open(f'{COCO_DIR}/annotations/instances_val2017.json')   as f: v = json.load(f)
id_to_name = {c['id']: c['name'] for c in t['categories']}
train_boxes = Counter(id_to_name[a['category_id']] for a in t['annotations'])
val_boxes   = Counter(id_to_name[a['category_id']] for a in v['annotations'])
rows = []
for cls in CLASSES:
    rows.append([cls, train_boxes.get(cls, 0), val_boxes.get(cls, 0)])
rows.append(['TOTAL', sum(train_boxes.values()), sum(val_boxes.values())])
print(tabulate(rows, headers=['class', 'train boxes', 'val boxes'], tablefmt='simple_outline'))

for cls, n in val_boxes.items():
    if n < 10:
        print(f'⚠  Class "{cls}" has only {n} val boxes — mAP for this class will be noisy.')
if sum(val_boxes.values()) < 30:
    print('⚠  Fewer than 30 total val boxes — expect noisy overall mAP. Annotate more val examples for stable metrics.')

## 8. Write the YOLOX experiment file

YOLOX is configured via a Python `Exp` class — subclass `yolox.exp.Exp` and override the hyperparameters. The trainer reads this file at runtime.

In [ ]:
EXP_PY = f'''
import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = {DEPTH}
        self.width = {WIDTH}
        self.num_classes = {NUM_CLASSES}
        self.exp_name = "{EXPERIMENT_NAME}"
        self.data_dir = "/content/coco"
        self.train_ann = "instances_train2017.json"
        self.val_ann   = "instances_val2017.json"
        # Training schedule
        self.max_epoch = {EPOCHS}
        self.no_aug_epochs = {NO_AUG_EPOCHS}
        self.warmup_epochs = 1
        self.eval_interval = 5
        # Augmentation
        self.mosaic_prob = {MOSAIC_PROB}
        self.mixup_prob = {MIXUP_PROB}
        self.enable_mixup = True
        # Optimization
        self.basic_lr_per_img = {BASIC_LR}
        self.input_size = ({IMG_SIZE}, {IMG_SIZE})
        self.test_size  = ({IMG_SIZE}, {IMG_SIZE})
'''
with open('/content/exp.py', 'w') as f:
    f.write(EXP_PY)
print('Wrote /content/exp.py')

## 9. Train

Runs Megvii YOLOX's training script as a subprocess. Progress streams to the notebook output. Per-epoch logs appear; mAP is evaluated every 5 epochs (controlled by `eval_interval` in the exp file).

On a T4 with `yolox_s`, batch 16, 640x640: roughly 25-50 seconds per epoch on a ~150-image dataset. 60 epochs ≈ 30-50 minutes total.

In [ ]:
import os
# Download the pretrained backbone weights from Megvii's public release.
# Each variant has its own .pth on GitHub. Resume-on-disk: skip download
# if a checkpoint of the same name is already present.
PRETRAINED_URLS = {
    'yolox_nano': 'https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth',
    'yolox_tiny': 'https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth',
    'yolox_s':    'https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth',
    'yolox_m':    'https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_m.pth',
    'yolox_l':    'https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_l.pth',
    'yolox_x':    'https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_x.pth',
}
PRETRAINED_PATH = f'/content/{BACKBONE}.pth'
if not os.path.exists(PRETRAINED_PATH):
    !wget -q -O "$PRETRAINED_PATH" "{PRETRAINED_URLS[BACKBONE]}"
print(f'Pretrained backbone: {PRETRAINED_PATH}  ({os.path.getsize(PRETRAINED_PATH) / 1e6:.1f} MB)')

# Launch training. yolox.tools.train runs an in-process training loop;
# we shell out so its stdout streams cleanly into the notebook output.
!cd /content && python -m yolox.tools.train -f /content/exp.py -d 1 -b {BATCH_SIZE} -c "$PRETRAINED_PATH" --fp16 -o

# Best checkpoint lives at YOLOX_outputs/<exp_name>/best_ckpt.pth
BEST_CKPT = f'/content/YOLOX_outputs/{EXPERIMENT_NAME}/best_ckpt.pth'
print(f'\nBest checkpoint: {BEST_CKPT}')
print(f'Exists: {os.path.exists(BEST_CKPT)}')

## 10. Evaluate

YOLOX's training already evaluated periodically and stamped per-epoch mAP into the log. The cell below reads the log and reports the final + best numbers.

In [ ]:
import re, os
log = f'/content/YOLOX_outputs/{EXPERIMENT_NAME}/train_log.txt'
if not os.path.exists(log):
    print('train_log.txt not found — training may have aborted; scroll up to the previous cell for the error.')
else:
    with open(log) as f: txt = f.read()
    # YOLOX writes 'Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.123'
    pattern = re.compile(r'Average Precision[^|]*\| area=\s*all\s*\| maxDets=100\s*\] = ([0-9.]+)')
    matches = pattern.findall(txt)
    if matches:
        best = max(float(m) for m in matches)
        final = float(matches[-1])
        print(f'  Best mAP @ 0.5:0.95 : {best:.3f}')
        print(f'  Final mAP @ 0.5:0.95: {final:.3f}')
    else:
        print('No mAP entries found in log — training may not have hit an evaluation interval.')
    # Also surface mAP @ 0.5 (less strict, easier to read for small datasets)
    pat50 = re.compile(r'Average Precision[^|]*\| IoU=0\.50\s*\| area=\s*all\s*\| maxDets=100\s*\] = ([0-9.]+)')
    m50 = pat50.findall(txt)
    if m50:
        print(f'  Best mAP @ 0.5      : {max(float(m) for m in m50):.3f}')

## 11. Export to ONNX

Converts the trained PyTorch model to ONNX with the contract IGNODE's inference runtime expects:

- Opset 18 (matches IGNODE's pinned ONNX stack)
- Dynamic batch dimension
- NMS embedded in the graph (IGNODE's `yolox_with_nms` decoder consumes this shape directly)

If NMS-embedded export fails on your environment, the cell falls back to a no-NMS export — IGNODE has a separate decoder (`yolox_raw`) that does NMS on the inference side.

In [ ]:
import os, subprocess
ONNX_PATH = '/content/model.onnx'
EXPORT_ARGS = [
    'python', '-m', 'yolox.tools.export_onnx',
    '-f', '/content/exp.py',
    '-c', BEST_CKPT,
    '--output-name', ONNX_PATH,
    '--opset', '18',
    '--dynamic',
    '--decode_in_inference',   # embed NMS-like decode into the graph
]
result = subprocess.run(EXPORT_ARGS, capture_output=True, text=True)
if result.returncode != 0:
    print('NMS-embedded export failed — falling back to no-NMS shape (compatible with IGNODE yolox_raw decoder)')
    print(result.stderr[-500:])
    EXPORT_ARGS = [a for a in EXPORT_ARGS if a != '--decode_in_inference']
    result = subprocess.run(EXPORT_ARGS, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Both ONNX export paths failed: {result.stderr}')

size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
print(f'✅ ONNX written: {ONNX_PATH}  ({size_mb:.1f} MB)')

# Sanity check the export
import onnxruntime as ort, numpy as np
session = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
input_meta = session.get_inputs()[0]
output_meta = session.get_outputs()[0]
print(f'   Input:  {input_meta.name} {input_meta.shape}')
print(f'   Output: {output_meta.name} {output_meta.shape}')
dummy = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
out = session.run(None, {input_meta.name: dummy})[0]
print(f'   Sanity: input {dummy.shape} → output {out.shape}')

## 12. Write sidecar files

IGNODE's inference runtime reads two sidecars alongside the model:

- `class_labels.json` — class names in the order the model emits them
- `preprocess_config.json` — input size, letterbox config, channel order. Tells the inference pod to preprocess images at predict time exactly the way we did at training time.

In [ ]:
import json

with open('/content/class_labels.json', 'w') as f:
    json.dump(CLASSES, f, indent=2)

preprocess = {
    'input_size':    [IMG_SIZE, IMG_SIZE],
    'channel_order': 'RGB',
    'image_format':  'CHW',
    'resize_method': 'letterbox',
    'pad_value':     114,
    'rescale':       'none',           # YOLOX takes raw 0-255 pixel values
    'normalize':     False,            # explicit — no ImageNet mean/std
    '_backbone':     BACKBONE,         # audit only; IGNODE's loader ignores this
    '_task':         'image_detection',
}
with open('/content/preprocess_config.json', 'w') as f:
    json.dump(preprocess, f, indent=2)

print('class_labels.json:')
print(json.dumps(CLASSES, indent=2))
print('\npreprocess_config.json:')
print(json.dumps(preprocess, indent=2))

## 13. Download

In [ ]:
import zipfile
from datetime import datetime
from google.colab import files

ZIP_NAME = f'ignode-image-detector-{datetime.now().strftime("%Y%m%d-%H%M")}.zip'
ARTIFACTS = ['/content/model.onnx', '/content/class_labels.json', '/content/preprocess_config.json']

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in ARTIFACTS:
        zf.write(path, arcname=path.split('/')[-1])

print(f'Bundled {len(ARTIFACTS)} files into {ZIP_NAME}')
files.download(ZIP_NAME)

## 14. Upload to IGNODE

1. **Integrations → ML Factory** in your IGNODE portal
2. Switch to the **Custom Models** tab
3. Click **+ Upload ML Model** (or the **Add Model** dropdown → **Upload Custom Model**)
4. Unzip the bundle, then drop your `model.onnx` and fill in the metadata form:
    - **Task Type:** `Image Detection`
    - **Class Labels:** paste from `class_labels.json`
    - **Image Preprocessing:** values from `preprocess_config.json` (the wizard auto-detects most of these, but verify they match)
5. Click **Upload**, then **Open in Playground** to test against a held-out image.

Image-detection models can only deploy to **ML Inference (UINF)** — Workspaces don't support image inference. The wizard will hide the Workspace target option.

---

## Reusing this notebook for your own data

Two cells in Step 3 — use either A or B, not both. Most customers use A (drag-drop) for one-off training; B (URL paste) is convenient if your data is already in IGNODE or any other cloud storage with signed-URL support.

### Optional tweaks (Step 7 Settings cell)

| Want to change | Edit |
|---|---|
| Stronger backbone (more accurate, slower) | `BACKBONE = 'yolox_m'` or `'yolox_l'` (needs > 12 GB VRAM) |
| Smaller backbone (less likely to overfit on tiny data) | `BACKBONE = 'yolox_tiny'` or `'yolox_nano'` |
| More training | `EPOCHS = 100` |
| Earlier close-mosaic for small datasets | `NO_AUG_EPOCHS = 30` (out of 100 total) |
| Smaller batches (if OOM) | `BATCH_SIZE = 8` |
| Smaller input (faster training, weaker on small objects) | `IMG_SIZE = 416` |

### Tips

- **Detection needs more data than classification.** Aim for at least 500 train images, ideally with 20+ box annotations per class in val. Below 200 images mAP @ 0.5 will usually plateau at 5-15% no matter what you tune.
- **Phantom classes (declared but never annotated) drag mAP down** — this notebook drops them automatically in Step 5. If you see classes you expected disappearing, check your annotations.
- **Loss decreasing while mAP stays at 0** is normal for the first 15-30 epochs of detection training. The model has to cross a confidence + localization threshold before any predictions pass IoU 0.5. Don't stop early.
- **One backbone size up usually beats one architecture change.** `yolox_m` on the same data is often a bigger lift than switching to a different YOLO family.
- **Find more datasets to try on Roboflow Universe** (https://universe.roboflow.com) — thousands of free, pre-annotated detection datasets, downloadable as Roboflow COCO. They drop into this notebook unchanged.

### Common errors

| Error | Fix |
|---|---|
| `CUDA out of memory` | Reduce `BATCH_SIZE` to 8 or 4; or pick a smaller backbone (`yolox_tiny` / `yolox_nano`) |
| `Could not detect format` (Step 4) | Your archive layout doesn't match Roboflow COCO / Roboflow YOLO / MS-COCO. Check the previous cell's output — your dataset may be wrapped in an extra top-level folder |
| mAP stays at 0 after 60 epochs | Either your dataset is too small (< 200 images with < 20 boxes per class in val), or your annotations don't actually match the visible objects. Inspect a few annotated images manually. |
| Upload to IGNODE rejected: `image_deploy_workspace_unsupported` | You picked a Workspace target — image-detection models deploy to ML Inference (UINF) only. Re-select the target in the wizard. |

### Bringing this code into your own project

The trainer is Megvii YOLOX (Apache 2.0). The Colab-specific pieces are `google.colab.files.upload()` and `google.colab.files.download()`. To run outside Colab:

1. Replace the upload block in Step 3 with `tarfile.open('/your/local/dataset.tar').extractall(EXTRACT_DIR)` or similar
2. Replace `files.download(ZIP_NAME)` in Step 13 with a file move / S3 put / whatever your project does
3. Everything else is portable